## Introduction

In this notebook, we'll show you how to do user registration. This is helpful if you want to provide your customers specific experience by creating your own app. You'll need to implement this in your app.

There are 3 kinds of identity can be used to create an user or sign in:

* via e-mail
* via mobile phone number
* via Facebook

## E-mail Flow

Let's start from e-mail. Below chart is the registration flow via e-mail.

![E-mail regration flow](https://mtm.tg3ds.com/tutorials/images/register_by_email.png)

### Preparation

At the first beginning, we need to set the API key and import necessary libraries.

In [0]:
import requests
import json

APIKEY = 'YOUR_APIKEY'

And in this tutorial, we also need some data for operation. Please change the values of these variables based on your information.

In [0]:
PROVIDER = 99999 # CHANGE TO YOUR PROVIDER ID
EMAIL = 'THE_EMAIL_USED_TO_REGISTER'
PASSWORD = 'THE_PASSWORD_OF_THE_USER' # password must longer than 4 bytes

### Existence Check

Before you create an user, you should check whether the user exists or not first.

In [0]:
data = {'username': EMAIL, 'provider': PROVIDER}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/check_account?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to check account')
else:
  print('Is account available: ', resp.json()['available'])
  if not resp.json()['available']:
    print('Account already exists')

### Account Creation

Once we confirm the account doesn't exist, we can continue to create.

In [0]:
data = {'email': EMAIL, 'password': PASSWORD, 'provider': PROVIDER}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/register_email?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 201:
  print('Failed to register account, error: ', resp.text)
else:
  print('Register successfully')

### Sign-in

Since we got a new account, let's take a look at how to get access token for later operations. As descibed in the flow chart. First step is to sign-in to get auth token.

In [0]:
data = {'username': EMAIL, 'password': PASSWORD, 'provider': PROVIDER}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/signin?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to signin account, error: ', resp.text)
else:
  auth_token = resp.json()['auth_token']
  print('Auth token: ', auth_token)

### Access Token Retrieval

Then use returned auth token to get access token. Because the access token has expiration time, you can repeat this call to get a new one.

In [0]:
data = {'username': EMAIL, 'provider': PROVIDER, 'auth_token': auth_token}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/auth?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to auth, error: ', resp.text)
else:
  access_token = resp.json()['access_token']
  expired_in = resp.json()['expired_in']
  print('Access token: %s, and will expire after %d seconds' % (access_token, expired_in))

Now you can use access token to do other operations. To know more about those operations, you can refer to other tutorials. And if you're interested in rest registration flows, please continue to following sections.

## Mobile Phone Number Flow

The second flow is via mobile phone number. Following is the registration flow chart. It's only a little different from e-mail. There is two more steps before you create an account. You need to call "send phone confirmation code" and "confirm phone number" APIs first. The phone number used to create account will get a SMS message with the confirmation code.

![Mobile phone number regration flow](https://mtm.tg3ds.com/tutorials/images/register_by_mobile_phone.png)

### Preparation and Existence Check

Like the e-mail flow, set up the mobile phone number and check whether the user exists or not. *Remember to use phone number as `username` to check*.

In [0]:
MOBILE_PHONE_NUMBER = 'PHONE_NUMBER_OF_THE_USER' # Like +886912345678
PASSWORD = 'PASSWORD_OF_THE_USER'

data = {'username': MOBILE_PHONE_NUMBER, 'provider': PROVIDER}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/check_account?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to check account')
else:
  if not resp.json()['available']:
    print('Account already exists')
  else:
    print('Is account available: ', resp.json()['available'])

### Confirmation Code

Then we call the "send confirmation code" API. You can control the language of the SMS message by the parameter `locale`. Supported languages are listed in the API document.

In [0]:
data = {'mobile_phone': MOBILE_PHONE_NUMBER, 'provider': PROVIDER, 'locale': 'en'}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/req_confirmation_token?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to request confirmation code, error: ', resp.text)
else:
  print('Send confirmation code successfully')

Note that this API has abuse protection, so you'll need to wait for a specific time duration for next request. You may refer to API document for detail explanation.

Once you get the confirmation token, please change the variable below then we can call "confirm phone number" API to mark this phone number is confirmed.

In [0]:
CONFIRM_TOKEN = 'SET_RECEIVED_TOKEN_HERE'

data = {'mobile_phone': MOBILE_PHONE_NUMBER, 'provider': PROVIDER, 'confirm_token': CONFIRM_TOKEN}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/confirm_phone?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to confirm phone number, error: ', resp.text)
else:
  print('Confirm phone number successfully')

### Account Creation

Last step, let's create the user account.

In [0]:
data = {'mobile_phone': MOBILE_PHONE_NUMBER, 'password': PASSWORD, 'provider': PROVIDER, 'confirm_token': CONFIRM_TOKEN}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/register_phone?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 201:
  print('Failed to register account, error: ', resp.text)
else:
  print('Register successfully')

As e-mail, you should see the auth token in response and that's all. Sign-in and access token retrieval are the same as e-mail flow.

## Facebook Flow

The last identity type is Facebook. Below is the flow chart. To use Facebook account as identity, users don't have to register, but you need to bind your app to Facebook, request data access permission and follow [their flow](https://developers.facebook.com/docs/facebook-login) to get the access token.

![Mobile phone number regration flow](https://mtm.tg3ds.com/tutorials/images/signin_by_sns_token.png)

Okay, now let's directly go to the sign-in step. Please replace the value of variable `FB_ACCESS_TOKEN` with what you got from Facebook. (You may refer to next section to get an access token of a test user. It's easier)

In [0]:
FB_ACCESS_TOKEN = 'ACCESS_TOKEN_FROM_FB'
FB_PROVIDER = 1 # Facebook provider is 1

data = {'token': FB_ACCESS_TOKEN, 'provider': FB_PROVIDER}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/signin_sns?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to signin account, error: ', resp.text)
else:
  username = resp.json()['username']
  auth_token = resp.json()['auth_token']
  print('Get username: %s, auth token: %s' % (username, auth_token))


Then the authentication step is a little different from others. You need to pass three parameters. Field `username` and `auth_token` are from sign-in response.

In [0]:
data = {'username': username, 'provider': FB_PROVIDER, 'auth_token': auth_token}
headers = {'Content-Type': 'application/json'}
resp = requests.post('https://api.tg3ds.com/api/v1/users/auth?apikey=%s' % (APIKEY), data=json.dumps(data), headers=headers)
if resp.status_code != 200:
  print('Failed to authorize, error: ', resp.text)
else:
  access_token = resp.json()['access_token']
  expired_in = resp.json()['expired_in']
  print('Get access token: %s, and will expire after %d seconds' % (access_token, expired_in))


### Facebook Test Users

If you didn't build up your app yet or you want to do some tests, you can [create test users on Facebook app page](https://developers.facebook.com/docs/apps/test-users) or use following approach to create test users to try.

In [0]:
FB_APP_ID = 'YOUR_APP_ID'
FB_APP_SECRET = 'YOUR_APP_SECRET'

resp = requests.get('https://graph.facebook.com/oauth/access_token?client_id=%s&client_secret=%s&grant_type=client_credentials' % (FB_APP_ID, FB_APP_SECRET))
if resp.status_code != 200:
  print('Error: ', resp.status_code, ' ', resp.text)
else:
  app_token = resp.json()['access_token']
  print('FB App token: ', app_token)

  resp = requests.post('https://graph.facebook.com/v3.1/%s/accounts/test-users?access_token=%s' % (FB_APP_ID, app_token))
  if resp.status_code != 200:
    print('Error: ', resp.status_code, ' ', resp.text)
  else:
    result = resp.json()
    print('User FB id: ', result['id'], 'User FB access token: ', result['access_token'])

These are the registration and sign-in flow of all three kinds of identities.

Copyright 2019 TG3D Studio